# 🧠 Deep Learning Master Crash Course
> **From Perceptrons to Transformers in PyTorch**

Welcome to the **Deep Learning Master Crash Course**! This notebook provides a complete hands-on journey through Deep Learning. You will learn the theoretical math, implement core components from scratch in NumPy, and build state-of-the-art models using **PyTorch**.

---

### 📖 Table of Contents
1. [Module 1: Foundations of Deep Learning & Perceptrons](#module-1)
2. [Module 2: Training Mechanics & Backpropagation](#module-2)
3. [Module 3: PyTorch Core Fundamentals](#module-3)
4. [Module 4: Convolutional Neural Networks (CNNs)](#module-4)
5. [Module 5: Sequential Models & Recurrent Neural Networks (RNNs)](#module-5)
6. [Module 6: Modern Deep Learning & Intro to Transformers](#module-6)
7. [Summary & Best Practices Checklist](#summary)


In [ ]:
# Environment Setup & Utility Imports
import os
import random
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# Optional seaborn styling if available
try:
    import seaborn as sns
    sns.set_theme(style="whitegrid")
except ImportError:
    plt.style.use("ggplot")

# Set random seed for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

# Select Device (GPU if available, else CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"PyTorch Version: {torch.__version__}")


<a id="module-1"></a>
## 1. Foundations of Deep Learning & Perceptrons

### 1.1 ML vs. Deep Learning
- **Classical ML**: Requires manual feature engineering (e.g., extracting SIFT features, HOG, hand-crafted tabular metrics).
- **Deep Learning**: Learns hierarchical feature representations directly from raw data (pixels, audio waveforms, text tokens).

### 1.2 The Artificial Neuron (Perceptron)
The single perceptron takes inputs $X = [x_1, x_2, \dots, x_d]^T$, applies weights $W = [w_1, w_2, \dots, w_d]^T$ and bias $b$, then passes the linear combination through an activation function $\sigma$:

$$z = W^T X + b = \sum_{i=1}^{d} w_i x_i + b$$
$$\hat{y} = \sigma(z)$$

### 1.3 Activation Functions
Activation functions introduce **non-linearity**, allowing neural networks to learn complex non-linear decision boundaries.

1. **Sigmoid**: $\sigma(z) = \frac{1}{1 + e^{-z}} \in (0, 1)$
2. **Tanh**: \tanh(z) = \frac{e^z - e^{-z}}{e^z + e^{-z}} \in (-1, 1)
3. **ReLU (Rectified Linear Unit)**: \text{ReLU}(z) = \max(0, z)
4. **Leaky ReLU**: \text{LeakyReLU}(z) = \max(\alpha z, z) \quad (\alpha \approx 0.01)
5. **Softmax** (Multi-class output): \text{Softmax}(z_i) = \frac{e^{z_i}}{\sum_{j} e^{z_j}}


In [ ]:
# Visualizing Activation Functions and their Derivatives

z = np.linspace(-5, 5, 200)

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_grad(z):
    s = sigmoid(z)
    return s * (1 - s)

def relu(z):
    return np.maximum(0, z)

def relu_grad(z):
    return (z > 0).astype(float)

def leaky_relu(z, alpha=0.1):
    return np.where(z > 0, z, alpha * z)

def leaky_relu_grad(z, alpha=0.1):
    return np.where(z > 0, 1.0, alpha)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Sigmoid & Derivative
axes[0].plot(z, sigmoid(z), label="Sigmoid", color="blue", lw=2)
axes[0].plot(z, sigmoid_grad(z), label="Sigmoid Grad", color="blue", linestyle="--", lw=2)
axes[0].set_title("Sigmoid & Gradient (Vanishing Gradient Issue)")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

# ReLU & Derivative
axes[1].plot(z, relu(z), label="ReLU", color="green", lw=2)
axes[1].plot(z, relu_grad(z), label="ReLU Grad", color="green", linestyle="--", lw=2)
axes[1].set_title("ReLU & Gradient")
axes[1].grid(True, alpha=0.3)
axes[1].legend()

# Leaky ReLU
axes[2].plot(z, leaky_relu(z), label="Leaky ReLU", color="purple", lw=2)
axes[2].plot(z, leaky_relu_grad(z), label="Leaky ReLU Grad", color="purple", linestyle="--", lw=2)
axes[2].set_title("Leaky ReLU & Gradient")
axes[2].grid(True, alpha=0.3)
axes[2].legend()

plt.tight_layout()
plt.show()


<a id="module-2"></a>
## 2. Training Mechanics & Backpropagation

### 2.1 Loss Functions
The loss function $\mathcal{L}(y, \hat{y})$ measures the discrepancy between true target $y$ and predicted $\hat{y}$.

- **Mean Squared Error (MSE)** (Regression):
  $$\mathcal{L}_{\text{MSE}} = \frac{1}{N} \sum_{i=1}^N (y_i - \hat{y}_i)^2$$

- **Binary Cross-Entropy (BCE)** (Binary Classification):
  $$\mathcal{L}_{\text{BCE}} = -\frac{1}{N} \sum_{i=1}^N \left[ y_i \log(\hat{y}_i) + (1 - y_i) \log(1 - \hat{y}_i) \right]$$

- **Categorical Cross-Entropy (CCE)** (Multi-class Classification):
  $$\mathcal{L}_{\text{CCE}} = -\sum_{c=1}^C y_c \log(\hat{y}_c)$$

---

### 2.2 Optimization Algorithms
Weights are updated in the opposite direction of the gradient of loss w.r.t parameters $\theta$:

1. **Stochastic Gradient Descent (SGD)**:
   $$\theta \leftarrow \theta - \eta \nabla_\theta \mathcal{L}$$
2. **SGD with Momentum**:
   $$v_t = \beta v_{t-1} + (1-\beta) \nabla_\theta \mathcal{L}$$
   $$\theta \leftarrow \theta - \eta v_t$$
3. **Adam (Adaptive Moment Estimation)**:
   Combines First Moment (Momentum) and Second Moment (RMSprop) with bias correction for extremely robust performance.

---

### 2.3 Backpropagation (Chain Rule)
Backpropagation computes $\frac{\partial \mathcal{L}}{\partial W^{(l)}}$ for every layer $l$ using the chain rule of calculus:

$$\frac{\partial \mathcal{L}}{\partial W^{(1)}} = \frac{\partial \mathcal{L}}{\partial \hat{y}} \cdot \frac{\partial \hat{y}}{\partial z^{(2)}} \cdot \frac{\partial z^{(2)}}{\partial a^{(1)}} \cdot \frac{\partial a^{(1)}}{\partial z^{(1)}} \cdot \frac{\partial z^{(1)}}{\partial W^{(1)}}$$

Let's build a **2-layer Neural Network completely from scratch in NumPy** to observe this step-by-step!


In [ ]:
# Implementing a 2-Layer Neural Network from Scratch in NumPy

class TwoLayerNNNumPy:
    def __init__(self, input_dim, hidden_dim, output_dim):
        # He initialization for ReLU hidden layer
        self.W1 = np.random.randn(input_dim, hidden_dim) * np.sqrt(2.0 / input_dim)
        self.b1 = np.zeros((1, hidden_dim))
        
        # Xavier initialization for Sigmoid output layer
        self.W2 = np.random.randn(hidden_dim, output_dim) * np.sqrt(1.0 / hidden_dim)
        self.b2 = np.zeros((1, output_dim))
        
    def forward(self, X):
        # Layer 1
        self.z1 = np.dot(X, self.W1) + self.b1
        self.a1 = np.maximum(0, self.z1) # ReLU
        
        # Layer 2
        self.z2 = np.dot(self.a1, self.W2) + self.b2
        self.a2 = 1 / (1 + np.exp(-self.z2)) # Sigmoid
        return self.a2
    
    def backward(self, X, y, lr=0.1):
        m = X.shape[0]
        
        # Derivative of BCE loss w.r.t z2
        dz2 = self.a2 - y
        dW2 = np.dot(self.a1.T, dz2) / m
        db2 = np.sum(dz2, axis=0, keepdims=True) / m
        
        # Backprop into Layer 1
        da1 = np.dot(dz2, self.W2.T)
        dz1 = da1 * (self.z1 > 0) # ReLU derivative
        dW1 = np.dot(X.T, dz1) / m
        db1 = np.sum(dz1, axis=0, keepdims=True) / m
        
        # Gradient Descent Updates
        self.W2 -= lr * dW2
        self.b2 -= lr * db2
        self.W1 -= lr * dW1
        self.b1 -= lr * db1
        
    def compute_loss(self, y_pred, y_true):
        # BCE Loss
        epsilon = 1e-15
        y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
        return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

# Generate non-linear Moons dataset
X_moons, y_moons = make_moons(n_samples=600, noise=0.2, random_state=42)
y_moons = y_moons.reshape(-1, 1)

# Train the NumPy Neural Network
nn_numpy = TwoLayerNNNumPy(input_dim=2, hidden_dim=16, output_dim=1)
losses = []

for epoch in range(1000):
    y_pred = nn_numpy.forward(X_moons)
    loss = nn_numpy.compute_loss(y_pred, y_moons)
    losses.append(loss)
    nn_numpy.backward(X_moons, y_moons, lr=0.5)

print(f"Final NumPy NN Loss after 1000 epochs: {losses[-1]:.4f}")

# Plot Decision Boundary
xx, yy = np.meshgrid(np.linspace(-1.5, 2.5, 200), np.linspace(-1.0, 1.5, 200))
grid = np.c_[xx.ravel(), yy.ravel()]
probs = nn_numpy.forward(grid).reshape(xx.shape)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(losses, color="navy", lw=2)
plt.title("NumPy Neural Net Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("BCE Loss")
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.contourf(xx, yy, probs, levels=20, cmap="Spectral", alpha=0.8)
plt.scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons.ravel(), cmap="Spectral", edgecolors="k")
plt.title("Learned Decision Boundary (NumPy From Scratch)")
plt.tight_layout()
plt.show()


<a id="module-3"></a>
## 3. PyTorch Core Fundamentals

PyTorch is the premier deep learning framework, built around **Dynamic Computational Graphs** and GPU acceleration.

Key Concepts:
1. `torch.Tensor`: Multi-dimensional array with autograd support.
2. `torch.autograd`: Automatic derivative calculation engine (`loss.backward()`).
3. `torch.nn.Module`: Base class for defining network architectures.
4. `DataLoader`: Batches, shuffles, and parallelizes data loading.

Let's build a modular Multi-Layer Perceptron (MLP) in PyTorch with training and validation monitoring.


In [ ]:
# PyTorch MLP Implementation

class PyTorchMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, dropout_rate=0.2):
        super(PyTorchMLP, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(hidden_dim, output_dim)
        )
        
    def forward(self, x):
        return self.net(x)

# Prepare Dataset in PyTorch format
X_train, X_val, y_train, y_val = train_test_split(X_moons, y_moons, test_size=0.2, random_state=42)

X_train_t = torch.FloatTensor(X_train).to(device)
y_train_t = torch.FloatTensor(y_train).to(device)
X_val_t = torch.FloatTensor(X_val).to(device)
y_val_t = torch.FloatTensor(y_val).to(device)

train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# Instantiate Model, Loss, Optimizer
model_mlp = PyTorchMLP(input_dim=2, hidden_dim=32, output_dim=1).to(device)
criterion = nn.BCEWithLogitsLoss() # Numerically stable BCE + Sigmoid
optimizer = optim.Adam(model_mlp.parameters(), lr=0.01, weight_decay=1e-4)

# Training Loop
epochs = 150
train_losses, val_losses = [], []

for epoch in range(epochs):
    model_mlp.train()
    batch_losses = []
    for bx, by in train_loader:
        optimizer.zero_grad()
        logits = model_mlp(bx)
        loss = criterion(logits, by)
        loss.backward()
        optimizer.step()
        batch_losses.append(loss.item())
        
    train_loss = np.mean(batch_losses)
    train_losses.append(train_loss)
    
    # Validation
    model_mlp.eval()
    with torch.no_grad():
        val_logits = model_mlp(X_val_t)
        val_loss = criterion(val_logits, y_val_t).item()
        val_losses.append(val_loss)

print(f"Final PyTorch MLP Train Loss: {train_losses[-1]:.4f} | Val Loss: {val_losses[-1]:.4f}")

# Plot Training vs Validation Loss
plt.figure(figsize=(8, 4))
plt.plot(train_losses, label="Train Loss", color="teal", lw=2)
plt.plot(val_losses, label="Val Loss", color="coral", lw=2)
plt.title("PyTorch MLP Learning Curve")
plt.xlabel("Epoch")
plt.ylabel("BCE Logits Loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


<a id="module-4"></a>
## 4. Convolutional Neural Networks (CNNs)

### 4.1 Why CNNs for Grid/Image Data?
Fully-connected MLPs flatten 2D/3D images into 1D vectors, destroying spatial relationships. **CNNs** preserve spatial structure using local receptive fields.

### 4.2 Key Operations
1. **Convolution Layer**: Applies small kernels/filters ($3 \times 3, 5 \times 5$) across spatial dimensions.
   - Output spatial dimension equation:
     $$O = \left\lfloor \frac{W - K + 2P}{S} \right\rfloor + 1$$
     Where $W$ = Input width, $K$ = Kernel size, $P$ = Padding, $S$ = Stride.
2. **Pooling Layer**: Reduces spatial dimensions (e.g. Max Pooling $2 \times 2$ downsamples height & width by half).
3. **Receptive Field & Feature Maps**: Shallow layers detect edges/textures; deep layers detect complex shapes/objects.


In [ ]:
# PyTorch CNN for Image Classification & Feature Map Extraction

class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super(SimpleCNN, self).__init__()
        # Conv Block 1
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(16)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Conv Block 2
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)
        
        # FC Head
        self.fc1 = nn.Linear(32 * 7 * 7, 64)
        self.fc2 = nn.Linear(64, num_classes)
        
    def forward(self, x):
        # x shape: [B, 1, 28, 28]
        feat1 = self.pool(self.relu(self.bn1(self.conv1(x)))) # -> [B, 16, 14, 14]
        feat2 = self.pool(self.relu(self.bn2(self.conv2(feat1)))) # -> [B, 32, 7, 7]
        
        flat = feat2.view(feat2.size(0), -1) # Flatten
        h = self.relu(self.fc1(flat))
        out = self.fc2(h)
        return out, feat1, feat2

# Create Synthetic 28x28 Image Data
synthetic_images = torch.randn(64, 1, 28, 28).to(device)
synthetic_labels = torch.randint(0, 10, (64,)).to(device)

cnn_model = SimpleCNN(num_classes=10).to(device)
logits, f1, f2 = cnn_model(synthetic_images)

print(f"Input Shape: {synthetic_images.shape}")
print(f"Layer 1 Feature Map Shape: {f1.shape}")
print(f"Layer 2 Feature Map Shape: {f2.shape}")
print(f"Output Logits Shape: {logits.shape}")

# Visualize learned 1st layer feature maps for 1 sample
sample_f1 = f1[0].detach().cpu().numpy()

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for idx, ax in enumerate(axes.flat):
    ax.imshow(sample_f1[idx], cmap="magma")
    ax.axis("off")
    ax.set_title(f"Filter {idx+1}")
plt.suptitle("CNN 1st Layer Feature Map Outputs", fontsize=14)
plt.tight_layout()
plt.show()


<a id="module-5"></a>
## 5. Sequential Models & Recurrent Neural Networks (RNNs)

### 5.1 Sequential Data & Vanilla RNNs
For sequential data (time-series, text, signals), state is maintained across time steps $t$:

$$h_t = \tanh(W_{hh} h_{t-1} + W_{xh} x_t + b_h)$$
$$y_t = W_{hy} h_t + b_y$$

- **Issue**: Standard RNNs suffer from **Vanishing and Exploding Gradients** when backpropagating through long time steps.

---

### 5.2 LSTM (Long Short-Term Memory)
LSTMs resolve vanishing gradients using a **Cell State** ($C_t$) regulated by 3 gating mechanisms:

1. **Forget Gate**: $f_t = \sigma(W_f \cdot [h_{t-1}, x_t] + b_f)$ (What to discard)
2. **Input Gate**: $i_t = \sigma(W_i \cdot [h_{t-1}, x_t] + b_i)$, $\tilde{C}_t = \tanh(W_c \cdot [h_{t-1}, x_t] + b_c)$ (What to update)
3. **Cell State Update**: $C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t$
4. **Output Gate**: $o_t = \sigma(W_o \cdot [h_{t-1}, x_t] + b_o)$, $h_t = o_t \odot \tanh(C_t)$


In [ ]:
# PyTorch LSTM for Time-Series / Sequence Prediction

class LSTMModel(nn.Module):
    def __init__(self, input_size=1, hidden_size=32, num_layers=1, output_size=1):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)
        
    def forward(self, x):
        # x shape: [Batch, Seq_Len, Input_Size]
        lstm_out, (hn, cn) = self.lstm(x)
        # Take the output of the last sequence step
        last_out = lstm_out[:, -1, :]
        prediction = self.fc(last_out)
        return prediction

# Generate Sine Wave Sequence Data
time_steps = np.linspace(0, 50, 500)
sine_wave = np.sin(time_steps)

seq_length = 20
X_seq, y_seq = [], []
for i in range(len(sine_wave) - seq_length):
    X_seq.append(sine_wave[i:i+seq_length])
    y_seq.append(sine_wave[i+seq_length])

X_seq = np.array(X_seq)[..., np.newaxis] # [N, seq_len, 1]
y_seq = np.array(y_seq)[..., np.newaxis]

X_seq_t = torch.FloatTensor(X_seq).to(device)
y_seq_t = torch.FloatTensor(y_seq).to(device)

# Train LSTM
lstm_net = LSTMModel(input_size=1, hidden_size=32, num_layers=1).to(device)
opt_lstm = optim.Adam(lstm_net.parameters(), lr=0.01)
crit_lstm = nn.MSELoss()

for ep in range(100):
    lstm_net.train()
    opt_lstm.zero_grad()
    preds = lstm_net(X_seq_t)
    loss = crit_lstm(preds, y_seq_t)
    loss.backward()
    opt_lstm.step()

lstm_net.eval()
with torch.no_grad():
    predictions = lstm_net(X_seq_t).cpu().numpy()

plt.figure(figsize=(12, 4))
plt.plot(y_seq, label="Ground Truth Sine Wave", color="black", lw=2)
plt.plot(predictions, label="LSTM Predictions", color="crimson", linestyle="--", lw=2)
plt.title("LSTM Sequence Forecasting Performance")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


<a id="module-6"></a>
## 6. Modern Deep Learning & Intro to Transformers

### 6.1 Attention Mechanism & Self-Attention
Traditional sequence models process tokens sequentially. **Self-Attention** computes direct pairwise relations between all tokens in parallel using Query ($Q$), Key ($K$), and Value ($V$) projections:

$$\text{Attention}(Q, K, V) = \text{softmax}\left( \frac{Q K^T}{\sqrt{d_k}} \right) V$$

Where $\sqrt{d_k}$ is the scaling factor to prevent small gradients when embedding dimension $d_k$ is large.

---

### 6.2 Transfer Learning
Rather than training deep networks from scratch, we use models pre-trained on large datasets (e.g. ImageNet) and fine-tune final linear layers on specialized target tasks.


In [ ]:
# 1. Implementing Scaled Dot-Product Self-Attention in PyTorch

class ScaledDotProductAttention(nn.Module):
    def __init__(self, d_model):
        super(ScaledDotProductAttention, self).__init__()
        self.d_k = d_model
        
    def forward(self, Q, K, V):
        # Q, K, V shapes: [Batch, Seq_Len, d_model]
        scores = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(self.d_k)
        attn_weights = torch.softmax(scores, dim=-1)
        output = torch.matmul(attn_weights, V)
        return output, attn_weights

# Demo Self-Attention
batch_size, seq_len, d_model = 2, 5, 16
Q = K = V = torch.randn(batch_size, seq_len, d_model)

attn_layer = ScaledDotProductAttention(d_model)
attn_output, weights = attn_layer(Q, K, V)

print(f"Self-Attention Output Shape: {attn_output.shape}")
print(f"Attention Weights Matrix Shape: {weights.shape}")

# Visualize Attention Matrix for 1 Sample using Matplotlib imshow
plt.figure(figsize=(6, 5))
plt.imshow(weights[0].detach().numpy(), cmap="Blues")
plt.colorbar(label="Attention Weight")
plt.title("Self-Attention Weights Matrix (Token-to-Token Relations)")
plt.xlabel("Key Tokens")
plt.ylabel("Query Tokens")
plt.show()


In [ ]:
# 2. Transfer Learning using Pretrained Torchvision Model

try:
    import torchvision.models as models
    resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

    # Freeze backbone parameters
    for param in resnet.parameters():
        param.requires_grad = False

    # Replace FC head for custom 5-class classification task
    num_ftrs = resnet.fc.in_features
    resnet.fc = nn.Linear(num_ftrs, 5)
    resnet = resnet.to(device)

    print("Pretrained ResNet18 modified for Transfer Learning:")
    print(f"Modified FC Layer: {resnet.fc}")
    print(f"Trainable Parameters count: {sum(p.numel() for p in resnet.parameters() if p.requires_grad)}")
except Exception as e:
    print(f"Torchvision transfer learning example note: {e}")


<a id="summary"></a>
## 7. Summary & Deep Learning Best Practices

### 🚀 Best Practices Checklist for Deep Learning Projects

1. **Data Preprocessing**:
   - Always scale/normalize inputs (e.g. Zero-mean, unit variance $\mu=0, \sigma=1$).
   - Shuffle training datasets; keep validation/test sets unshuffled.
2. **Architecture Selection**:
   - Tabular data $\rightarrow$ MLPs or Gradient Boosted Trees.
   - Image/Grid data $\rightarrow$ CNNs or Vision Transformers.
   - Sequential/Text data $\rightarrow$ LSTMs, GRUs, or Transformers.
3. **Regularization & Overfitting Prevention**:
   - Use Dropout ($0.1 - 0.5$) and Batch Normalization.
   - Apply Weight Decay ($L_2$ Regularization).
   - Implement **Early Stopping** based on validation loss.
4. **Learning Rate Management**:
   - Start with Adam default learning rate ($1e-3$ or $3e-4$).
   - Use Learning Rate Schedulers (`ReduceLROnPlateau`, `CosineAnnealingLR`).
5. **Debugging Neural Networks**:
   - Try overfitting a tiny batch of 5-10 samples first. If the model can't achieve ~0 loss on a tiny batch, there is a bug in model architecture, loss function, or data pipeline!

---
*End of Deep Learning Crash Course Notebook.*
